In [ ]:
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import Callback, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertTokenizer
from tensorflow.keras.optimizers import Adam 
from transformers import TFBertForSequenceClassification, TFBertModel
import tensorflow as tf
import tensorflow_addons as tfa


In [ ]:
# 1. Set Python hash seed (for consistency in hashing)
os.environ['PYTHONHASHSEED'] = '0'

# 2. Set seeds for Python, NumPy, and TensorFlow
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# 3. (Optional) Configure TensorFlow for deterministic operations
# This helps eliminate non-deterministic GPU ops
os.environ['TF_DETERMINISTIC_OPS'] = '1'

In [ ]:
train_data = pd.read_csv('train_data_no_dup1.csv')
val_data = pd.read_csv('val_data_no_dup1.csv')
test_data = pd.read_csv('test_data_no_dup1.csv')

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
test_data.shape

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
max_sample_length = max(train_data['NFR1'].apply(lambda x: len(x.split())))
print(f"Maximum sample length: {max_sample_length}")

max_sample_length = max(train_data['NFR2'].apply(lambda x: len(x.split())))
print(f"Maximum sample length: {max_sample_length}")

In [ ]:
def encode_pair(df, tokenizer):
    return tokenizer(
        list(df['NFR1']),
        list(df['NFR2']),
        truncation=True,
        padding='max_length',
        max_length=120,
        return_tensors='tf'
    )


In [ ]:
train_encodings = encode_pair(train_data, tokenizer)
val_encodings   = encode_pair(val_data, tokenizer)
test_encodings  = encode_pair(test_data, tokenizer)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_data['conflict status'].values
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_data['conflict status'].values
))

test_dataset = tf.data.Dataset.from_tensor_slices(dict(test_encodings))


In [ ]:
batch_size = 16
train_dataset = train_dataset.shuffle(buffer_size=len(train_dataset), reshuffle_each_iteration=True).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size)
test_dataset = test_dataset.batch(batch_size)

In [ ]:
# Load the pre-trained BERT model
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

In [ ]:
bert_model.bert.embeddings.trainable = False
bert_model.bert.pooler.trainable     = False
bert_model.bert.trainable = False 
# Freeze all encoder layers first
for layer in bert_model.bert.encoder.layer:
    layer.trainable = False

In [ ]:
for layer in bert_model.layers:
    for sublayer in layer.submodules:
        print(sublayer.name, sublayer.trainable)


In [ ]:
# Input layers
input_ids = tf.keras.layers.Input(shape=(120,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.layers.Input(shape=(120,), dtype=tf.int32, name="attention_mask")
token_type_ids = tf.keras.layers.Input(shape=(120,), dtype=tf.int32, name="token_type_ids")

# Get BERT outputs
bert_outputs = bert_model([input_ids, attention_mask, token_type_ids])
pooled_output = bert_outputs.pooler_output  # same as bert_outputs[1]

# Classification head
drop = tf.keras.layers.Dropout(0.1)(pooled_output)
output = tf.keras.layers.Dense(1, activation='sigmoid')(drop)  # binary classification

# Build full model
model = tf.keras.Model(
    inputs=[input_ids, attention_mask, token_type_ids],
    outputs=output
)

In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=[tfa.metrics.F1Score(num_classes=1, threshold=0.5, average='micro')])

# Show the model summary
model.summary()

In [ ]:
for layer in model.layers:
    print(layer.name, layer.trainable)


In [ ]:
checkpoint_filepath = 'best_model.keras'

In [ ]:
model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=False,  # Set to True if you only want to save weights, False to save the entire model
        monitor='val_f1_score', 
        mode='max',  
        save_best_only=True, 
        verbose=1  # Log a message whenever a new checkpoint is saved
    )

In [ ]:
# Train the model
epochs = 50  # You can adjust this

history = model.fit(train_dataset,
                    epochs=epochs,
                    validation_data=val_dataset,
                    callbacks=model_checkpoint_callback)


In [ ]:
sns.set()
fig = plt.figure(0, (12, 4))

ax = plt.subplot(1, 2, 1)
sns.lineplot( history.history['f1_score'], label='train')
sns.lineplot(history.history['val_f1_score'], label='valid')
plt.title('Accuracy')
plt.tight_layout()

ax = plt.subplot(1, 2, 2)
sns.lineplot(history.history['loss'], label='train')
sns.lineplot(history.history['val_loss'], label='valid')
plt.title('Loss')
plt.tight_layout()

plt.show()

In [ ]:
# Load weights from the saved model
model.load_weights(checkpoint_filepath)

In [ ]:
test_labels= test_data['conflict status'].values

In [ ]:
# Make predictions on the test set
test_predictions = model.predict(test_dataset)

# For binary classification, this would be a threshold of 0.5
predicted_classes = (test_predictions >= 0.5).astype("int32")

# Generate the classification report
print("\nClassification Report:")
print(classification_report(test_labels, predicted_classes, digits=4))

# Generate the confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(test_labels, predicted_classes)

# Plotting the confusion matrix
plt.figure(figsize=(7, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
            xticklabels=np.unique(train_data['conflict status']),
            yticklabels=np.unique(train_data['conflict status']))
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
bert_model.bert.embeddings.trainable = True
bert_model.bert.pooler.trainable     = True
bert_model.bert.trainable = True 
# Freeze all encoder layers first
for layer in bert_model.bert.encoder.layer:
    layer.trainable = True

In [ ]:
for layer in bert_model.layers:
    for sublayer in layer.submodules:
        print(sublayer.name, sublayer.trainable)


In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=[tfa.metrics.F1Score(num_classes=1, threshold=0.5, average='micro')])

# Show the model summary
model.summary()

In [ ]:
checkpoint_filepath2 = 'best_model2.keras'

In [ ]:
model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath2,
        save_weights_only=False,  # Set to True if you only want to save weights, False to save the entire model
        monitor='val_f1_score', 
        mode='max',   
        save_best_only=True, 
        verbose=1  # Log a message whenever a new checkpoint is saved
    )

In [ ]:
# Train the model
epochs = 10  # You can adjust this

history = model.fit(train_dataset,
                    epochs=epochs,
                    validation_data=val_dataset,
                    callbacks=model_checkpoint_callback)


In [ ]:
sns.set()
fig = plt.figure(0, (12, 4))

ax = plt.subplot(1, 2, 1)
sns.lineplot( history.history['f1_score'], label='train')
sns.lineplot(history.history['val_f1_score'], label='valid')
plt.title('Accuracy')
plt.tight_layout()

ax = plt.subplot(1, 2, 2)
sns.lineplot(history.history['loss'], label='train')
sns.lineplot(history.history['val_loss'], label='valid')
plt.title('Loss')
plt.tight_layout()

plt.show()

In [ ]:
# Load weights from the saved model
model.load_weights(checkpoint_filepath2)

In [ ]:
test_labels= test_data['conflict status'].values

In [ ]:
# Make predictions on the test set
test_predictions = model.predict(test_dataset)

# For binary classification, this would be a threshold of 0.5
predicted_classes = (test_predictions >= 0.5).astype("int32")

# Generate the classification report
print("\nClassification Report:")
print(classification_report(test_labels, predicted_classes, digits=4))

# Generate the confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(test_labels, predicted_classes)

# Plotting the confusion matrix
plt.figure(figsize=(7, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
            xticklabels=np.unique(train_data['conflict status']),
            yticklabels=np.unique(train_data['conflict status']))
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()
